# Step 1 — Data Cleaning: Online Retail Sales
**Project:** E-commerce Sales Analysis (UK online gift retailer, Dec 2010 – Dec 2011)

**Goal of this notebook:** turn the raw transaction file into a clean, trustworthy dataset that EDA, SQL and Power BI can all use.

**Why cleaning matters:** if cancelled orders, duplicates or postage charges stay in the data, every revenue number in the dashboard is wrong. Clients judge an analyst by whether the numbers can be trusted, so cleaning matters more than the charts.

**Outputs of this notebook:**
- `Data/Processed/online_retail_clean.csv`: clean sales lines (used for EDA, SQL, Power BI)
- `Data/Processed/online_retail_cancellations.csv`: cancelled orders (used to analyse returns)

## 1. Import libraries
- **pandas**: works with tables (DataFrames) and is the main tool in this notebook
- **numpy**: numeric helpers

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)   # show all columns when printing
pd.set_option('display.float_format', '{:.2f}'.format)

## 2. Load the raw data
- The file contains special characters (like `£` and accented letters), so it is **not** UTF-8. We read it with `encoding='ISO-8859-1'` (Latin-1), otherwise pandas throws a `UnicodeDecodeError`.
- We read `CustomerID` as **text** (`str`). It is an ID, not a number you add up. Read as a number it would become `17850.0`, which looks wrong in reports.
- We never edit the raw file. We load it, clean a copy, and save the result somewhere else, so you can always restart from the original.

In [ ]:
RAW_PATH = '../Data/Raw/Online Retail.csv'

raw = pd.read_csv(RAW_PATH, encoding='ISO-8859-1', dtype={'CustomerID': str})
df = raw.copy()          # work on a copy, keep `raw` untouched for comparison

print('Rows, Columns:', df.shape)
df.head()

## 3. First look at the data
`info()` shows each column's data type and how many non-empty values it has. `describe()` shows min/max/mean. This is where we **spot problems**:
- `InvoiceDate` is stored as text (`object`/`str`), so it must be converted to datetime
- `Quantity` has **negative** values (returns/cancellations)
- `UnitPrice` has **zero/negative** values

In [ ]:
df.info()

In [ ]:
df.describe()

## 4. Check missing values
`isnull().sum()` counts empty cells per column.

**What we find:**
- **CustomerID** is missing for about 25% of rows. These are often guest checkouts. The sale still happened, so we **keep these rows for revenue analysis** and put `'Guest'` in the gap. For customer-level analysis (RFM, repeat rate) we will filter them out later.
- **Description** is missing for about 1,454 rows. Almost all of these have price 0 (system/adjustment lines), so they disappear when we remove zero-price rows in Step 8.

In [ ]:
missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_%': (df.isnull().mean() * 100).round(2)
})
missing

## 5. Keep a cleaning log
Every cleaning step records how many rows it removed. You will put this table in your **report, PPT and GitHub README**. It shows a client exactly what you did, and that transparency builds trust.

In [ ]:
log = []
def log_step(step, before, after):
    log.append({'step': step, 'rows_before': before, 'rows_removed': before - after, 'rows_after': after})
    print(f'{step}: removed {before - after:,} rows -> {after:,} remain')

## 6. Remove exact duplicate rows
A duplicate row is an identical line repeated, usually from a system glitch. Left in, it **double-counts revenue**. `drop_duplicates()` keeps the first copy and removes the rest.

In [ ]:
print('Duplicate rows:', df.duplicated().sum())

before = len(df)
df = df.drop_duplicates()
log_step('Remove exact duplicates', before, len(df))

## 7. Fix data types
- **InvoiceDate → datetime.** Text dates can't be grouped by month or sorted correctly. The format in this file is `month/day/year hour:minute` (e.g. `12/1/2010 8:26` = 1 December 2010), so we give pandas the exact format and it can't confuse day and month.
- **InvoiceNo, StockCode → text**, removing extra spaces.

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='%m/%d/%Y %H:%M')
df['InvoiceNo'] = df['InvoiceNo'].astype(str).str.strip()
df['StockCode'] = df['StockCode'].astype(str).str.strip().str.upper()

print('Date range:', df['InvoiceDate'].min(), 'to', df['InvoiceDate'].max())
df.dtypes

## 8. Separate cancelled orders
In this dataset an invoice number starting with **`C`** means a cancellation (e.g. `C536379`), with a negative quantity.

We **don't throw these away**. We save them to their own file, because the **return/cancellation rate** is a valuable insight for the business (which products get returned most?). The main sales table keeps only real sales.

In [ ]:
is_cancel = df['InvoiceNo'].str.startswith('C')
cancellations = df[is_cancel].copy()
print('Cancelled lines:', len(cancellations))

before = len(df)
df = df[~is_cancel]
log_step('Move cancellations to separate file', before, len(df))

## 9. Remove invalid quantity and price
After removing cancellations, a few rows still have:
- **Quantity ≤ 0**: stock adjustments or damaged items, not sales
- **UnitPrice ≤ 0**: free items, bad-debt adjustments, or missing-description system lines

None of these are real customer sales, so they would distort averages like *average selling price*.

In [ ]:
print('Quantity <= 0 :', (df['Quantity'] <= 0).sum())
print('UnitPrice <= 0:', (df['UnitPrice'] <= 0).sum())

before = len(df)
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
log_step('Remove Quantity/UnitPrice <= 0', before, len(df))

print('Missing descriptions left:', df['Description'].isnull().sum())

## 10. Remove non-product lines (postage, fees, adjustments)
Real products have numeric stock codes (e.g. `85123A`). Some codes are **not products**:

| Code | Meaning |
|---|---|
| POST, DOT, C2 | Postage / carriage charges |
| M | Manual adjustment |
| D | Discount |
| S | Samples |
| BANK CHARGES, AMAZONFEE | Fees |
| CRUK | Charity donation (Cancer Research UK) |
| B | Bad debt adjustment |
| PADS | Pads to match cushions (internal) |
| GIFT_0001_xx | Gift vouchers |

Left in, a single `AMAZON FEE` line of £13,541 would show up as your "most expensive product". `DCGS...` codes **are** real products (party bags), so we keep them.

In [ ]:
non_product_codes = ['POST', 'DOT', 'C2', 'M', 'D', 'S', 'BANK CHARGES',
                     'AMAZONFEE', 'CRUK', 'B', 'PADS']

is_non_product = df['StockCode'].isin(non_product_codes) | df['StockCode'].str.startswith('GIFT_')
print(df.loc[is_non_product, 'StockCode'].value_counts())

before = len(df)
df = df[~is_non_product]
log_step('Remove non-product codes', before, len(df))

## 11. Handle extreme outliers
Look at the biggest orders. Two orders (80,995 and 74,215 units) look like typing mistakes, and each was **cancelled straight away** by the same customer. They are still in our sales table because we only removed the cancellation line. Kept in, they'd add about £245K of fake revenue.

**Rule:** remove a large order (≥ 10,000 units) if a cancellation exists for the same customer, product and quantity. We only remove what we can prove is wrong. Genuine big wholesale orders stay.

In [ ]:
df.nlargest(5, 'Quantity')[['InvoiceNo','StockCode','Description','Quantity','UnitPrice','CustomerID','InvoiceDate']]

In [ ]:
cancel_keys = cancellations.assign(Quantity=cancellations['Quantity'].abs())[['CustomerID','StockCode','Quantity']]

big = df[df['Quantity'] >= 10000].reset_index()
reversed_big = big.merge(cancel_keys, on=['CustomerID','StockCode','Quantity'], how='inner')['index']
print('Large orders that were fully reversed:', len(reversed_big))

before = len(df)
df = df.drop(index=reversed_big)
log_step('Remove large orders that were cancelled', before, len(df))

## 12. Clean text columns
- **Description:** remove extra spaces so `"WHITE MUG "` and `"WHITE MUG"` are the same product.
- **Country:** fix unclear names so Power BI's map recognises them (`EIRE` → `Ireland`, `RSA` → `South Africa`).
- **CustomerID:** fill missing values with `'Guest'` and add an `IsGuest` flag, so we can filter guests in or out easily in SQL and Power BI.

In [ ]:
df['Description'] = df['Description'].str.strip().str.replace(r'\s+', ' ', regex=True)

df['Country'] = df['Country'].replace({'EIRE': 'Ireland', 'RSA': 'South Africa'})

df['IsGuest'] = df['CustomerID'].isnull()
df['CustomerID'] = df['CustomerID'].fillna('Guest')

print(df['Country'].value_counts().head(10))
print('\nGuest lines:', df['IsGuest'].sum())

## 13. Feature engineering (new useful columns)
Raw data rarely has the exact columns a dashboard needs, so we create them once here. That keeps SQL queries and Power BI measures much simpler.

| New column | Why it's useful |
|---|---|
| `Revenue` = Quantity × UnitPrice | The main sales metric |
| `InvoiceDay` | Daily trend |
| `Year`, `Month`, `MonthName`, `YearMonth` | Monthly trend charts |
| `Quarter` | Quarterly comparison |
| `DayOfWeek` | Which weekday sells most |
| `Hour` | Peak shopping hours |

In [ ]:
df['Revenue']    = (df['Quantity'] * df['UnitPrice']).round(2)
df['InvoiceDay'] = df['InvoiceDate'].dt.date
df['Year']       = df['InvoiceDate'].dt.year
df['Month']      = df['InvoiceDate'].dt.month
df['MonthName']  = df['InvoiceDate'].dt.strftime('%b')
df['YearMonth']  = df['InvoiceDate'].dt.strftime('%Y-%m')
df['Quarter']    = 'Q' + df['InvoiceDate'].dt.quarter.astype(str)
df['DayOfWeek']  = df['InvoiceDate'].dt.day_name()
df['Hour']       = df['InvoiceDate'].dt.hour

df.head()

## 14. Final quality checks
Before saving, we prove the data is clean. These `assert` lines stop the notebook with an error if anything is wrong, which works as automatic testing.

In [ ]:
assert df.duplicated().sum() == 0, 'Duplicates found'
assert (df['Quantity'] > 0).all(), 'Non-positive quantity found'
assert (df['UnitPrice'] > 0).all(), 'Non-positive price found'
assert df['Description'].isnull().sum() == 0, 'Missing descriptions found'
assert not df['InvoiceNo'].str.startswith('C').any(), 'Cancellations still present'
print('All checks passed ✔')

print(f"\nRows: {len(df):,}")
print(f"Invoices: {df['InvoiceNo'].nunique():,}")
print(f"Customers (excl. guests): {df.loc[~df['IsGuest'], 'CustomerID'].nunique():,}")
print(f"Products: {df['StockCode'].nunique():,}")
print(f"Countries: {df['Country'].nunique()}")
print(f"Total revenue: £{df['Revenue'].sum():,.2f}")

## 15. Cleaning summary
Copy this table into your report and README.

In [ ]:
log_df = pd.DataFrame(log)
log_df.loc[len(log_df)] = ['TOTAL', len(raw), len(raw) - len(df), len(df)]
log_df

## 16. Save the clean files
- `index=False`: don't write pandas' row numbers as an extra column
- Cancellations also get a positive `ReturnQty` and `ReturnValue`, which makes return analysis easier

In [ ]:
cancellations['ReturnQty'] = cancellations['Quantity'].abs()
cancellations['ReturnValue'] = (cancellations['ReturnQty'] * cancellations['UnitPrice']).round(2)
cancellations['CustomerID'] = cancellations['CustomerID'].fillna('Guest')

df.to_csv('../Data/Processed/online_retail_clean.csv', index=False)
cancellations.to_csv('../Data/Processed/online_retail_cancellations.csv', index=False)

print('Saved clean data:', df.shape)
print('Saved cancellations:', cancellations.shape)

## ✅ Key takeaways from cleaning
1. About **25% of sales lines have no CustomerID**. We kept them for revenue but flagged them as guests. *Business recommendation: encourage account sign-up at checkout.*
2. **About 9,000 cancellation lines** were moved to a separate file for return-rate analysis.
3. **Postage, fees and adjustments** were removed so product metrics reflect real products only.
4. **Two mistyped bulk orders** (about £245K of fake revenue) were removed because they were immediately cancelled.
5. The data covers **1 Dec 2010 – 9 Dec 2011**. December 2011 is a **partial month**, so remember this in trend charts.

**Next step → 02_EDA.ipynb**